# Notebook 3: Fraud Model Training (XGBoost)

Trains an XGBoost fraud classifier on 12M transactions with 147 features. Training data is generated directly from the `FRAUD_TRANSACTIONS` table — no Dynamic Tables or Feature Store required for training.

---

### Design Choices

| Decision | Choice | Rationale |
|---|---|---|
| Warehouse | FRAUD_OFS_TRAIN_WH (SP-Opt MEDIUM, 6 cr/hr) | 256GB dedicated RAM. 12M x 147 features fits comfortably. |
| Why not Standard XLARGE | SP-Opt MEDIUM is 62% cheaper AND more memory | Standard XLARGE = 16 cr/hr, ~80GB usable vs 256GB dedicated |
| tree_method | 'hist' | 5-10x faster than 'exact' at this scale, minimal accuracy loss |
| scale_pos_weight | 2000 (inverse fraud rate) | Handles 0.05% imbalance without memory-expensive oversampling |
| Evaluation metric | AUC-PR | ROC-AUC is misleading at 0.05% — a model predicting "never fraud" gets 99.95% accuracy |
| MAX_CONCURRENCY_LEVEL | 1 | Training gets exclusive access to all 256GB RAM |

In [ ]:
from snowflake.snowpark.context import get_active_session

# snowflake.ml.modeling.xgboost wraps the open-source XGBoost library so that training
# runs natively inside Snowflake on a Snowpark-Optimized warehouse (dedicated RAM/CPU),
# rather than pulling data to a local machine. No data ever leaves Snowflake.
from snowflake.ml.modeling.xgboost import XGBClassifier

# Registry is the Snowflake Model Registry — a versioned store for trained ML models.
# It handles serialisation, schema inference, and model promotion (DEV → STAGING → PROD).
from snowflake.ml.registry import Registry
from snowflake.snowpark.functions import col, lit, when, datediff, current_timestamp
from snowflake.snowpark.functions import hour, dayofweek
import time

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA ML').collect()

# Resume the Snowpark-Optimized MEDIUM warehouse if it auto-suspended.
# Snowpark-Optimized warehouses have dedicated memory (256GB on MEDIUM) vs shared on Standard.
# This is important for XGBoost — it needs to hold the full 1M-row training set in RAM.
session.sql('ALTER WAREHOUSE FRAUD_OFS_TRAIN_WH RESUME IF SUSPENDED').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()
print('Context: FRAUD_DEMO_DEV.ML, Snowpark-Optimized MEDIUM warehouse')

## 3.1 Generate Training Dataset

Training features are computed directly from the transactions table using window aggregations. No Dynamic Tables or Feature Store needed for training — the Online FS is a serving layer only.

This approach eliminates the 24/7 DT warehouse cost entirely.

In [ ]:
print('Generating training dataset with full OFS feature parity...')
print('Windows: 1h, 6h, 24h, 48h, 7d. Entities: customer, merchant, DPAN, IP + profile.')
import time
start = time.time()

# This SQL builds a point-in-time correct training set matching the Online Feature Store schema.
#
# Feature parity is critical: the model must be trained on the same features it will receive
# at inference time (from the OFS CONTINUOUS feature views).
#
# Customer velocity: point-in-time correct via self-joins.
# Merchant / DPAN / IP: historical aggregates up to each transaction time.
# Profile: static attributes from DIM_CUSTOMERS joined at transaction time.
# Derived features: ratio features computed from base velocity (same logic as scoring service).
training_df = session.sql("""
    WITH
    -- Merchant velocity: all transactions before t for same merchant
    merch_agg AS (
        SELECT
            m.txn_id,
            COUNT(h.TRANSACTION_TS)                                    AS MERCHANT_TXN_NUM_L1H,
            COUNT(h6.TRANSACTION_TS)                                   AS MERCHANT_TXN_NUM_L6H,
            COUNT(h24.TRANSACTION_TS)                                  AS MERCHANT_TXN_NUM_L24H,
            COUNT(h48.TRANSACTION_TS)                                  AS MERCHANT_TXN_NUM_L48H,
            COUNT(h1wk.TRANSACTION_TS)                                 AS MERCHANT_TXN_NUM_L1WK,
            COALESCE(SUM(h.PURCHASE_AMOUNT), 0)                        AS MERCHANT_TXN_AMT_L1H,
            COALESCE(SUM(h24.PURCHASE_AMOUNT), 0)                      AS MERCHANT_TXN_AMT_L24H,
            COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)                     AS MERCHANT_TXN_AMT_L1WK,
            COALESCE(MAX(h24.PURCHASE_AMOUNT), 0)                      AS MERCHANT_MAX_TXN_L24H,
            COUNT(DISTINCT h.CUSTOMER_ID)                              AS MERCHANT_UNIQUE_CUST_L1H,
            COUNT(DISTINCT h24.CUSTOMER_ID)                            AS MERCHANT_UNIQUE_CUST_L24H,
            COUNT(DISTINCT h1wk.CUSTOMER_ID)                           AS MERCHANT_UNIQUE_CUST_L1WK
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS m
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h
            ON h.MERCHANT_ID = m.MERCHANT_ID AND h.TRANSACTION_TS >= DATEADD('hour', -1, m.TRANSACTION_TS) AND h.TRANSACTION_TS < m.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h6
            ON h6.MERCHANT_ID = m.MERCHANT_ID AND h6.TRANSACTION_TS >= DATEADD('hour', -6, m.TRANSACTION_TS) AND h6.TRANSACTION_TS < m.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
            ON h24.MERCHANT_ID = m.MERCHANT_ID AND h24.TRANSACTION_TS >= DATEADD('hour', -24, m.TRANSACTION_TS) AND h24.TRANSACTION_TS < m.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h48
            ON h48.MERCHANT_ID = m.MERCHANT_ID AND h48.TRANSACTION_TS >= DATEADD('hour', -48, m.TRANSACTION_TS) AND h48.TRANSACTION_TS < m.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
            ON h1wk.MERCHANT_ID = m.MERCHANT_ID AND h1wk.TRANSACTION_TS >= DATEADD('day', -7, m.TRANSACTION_TS) AND h1wk.TRANSACTION_TS < m.TRANSACTION_TS
        GROUP BY m.txn_id
    ),
    -- DPAN velocity: all transactions before t for same wallet DPAN
    dpan_agg AS (
        SELECT
            d.txn_id,
            COUNT(h.TRANSACTION_TS)              AS DPAN_TXN_NUM_L1H,
            COUNT(h6.TRANSACTION_TS)             AS DPAN_TXN_NUM_L6H,
            COUNT(h24.TRANSACTION_TS)            AS DPAN_TXN_NUM_L24H,
            COUNT(h48.TRANSACTION_TS)            AS DPAN_TXN_NUM_L48H,
            COUNT(h1wk.TRANSACTION_TS)           AS DPAN_TXN_NUM_L1WK,
            COALESCE(SUM(h24.PURCHASE_AMOUNT),0) AS DPAN_TXN_AMT_L24H,
            COALESCE(SUM(h1wk.PURCHASE_AMOUNT),0)AS DPAN_TXN_AMT_L1WK,
            COUNT(DISTINCT h24.CUSTOMER_ID)      AS DPAN_DISTINCT_CUST_L24H,
            COUNT(DISTINCT h1wk.CUSTOMER_ID)     AS DPAN_DISTINCT_CUST_L1WK
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS d
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h
            ON h.WALLET_DPAN = d.WALLET_DPAN AND h.TRANSACTION_TS >= DATEADD('hour', -1, d.TRANSACTION_TS) AND h.TRANSACTION_TS < d.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h6
            ON h6.WALLET_DPAN = d.WALLET_DPAN AND h6.TRANSACTION_TS >= DATEADD('hour', -6, d.TRANSACTION_TS) AND h6.TRANSACTION_TS < d.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
            ON h24.WALLET_DPAN = d.WALLET_DPAN AND h24.TRANSACTION_TS >= DATEADD('hour', -24, d.TRANSACTION_TS) AND h24.TRANSACTION_TS < d.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h48
            ON h48.WALLET_DPAN = d.WALLET_DPAN AND h48.TRANSACTION_TS >= DATEADD('hour', -48, d.TRANSACTION_TS) AND h48.TRANSACTION_TS < d.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
            ON h1wk.WALLET_DPAN = d.WALLET_DPAN AND h1wk.TRANSACTION_TS >= DATEADD('day', -7, d.TRANSACTION_TS) AND h1wk.TRANSACTION_TS < d.TRANSACTION_TS
        GROUP BY d.txn_id
    ),
    -- IP velocity: all transactions before t for same IP address
    ip_agg AS (
        SELECT
            i.txn_id,
            COUNT(h.TRANSACTION_TS)              AS IP_TXN_NUM_L1H,
            COUNT(h6.TRANSACTION_TS)             AS IP_TXN_NUM_L6H,
            COUNT(h24.TRANSACTION_TS)            AS IP_TXN_NUM_L24H,
            COUNT(h48.TRANSACTION_TS)            AS IP_TXN_NUM_L48H,
            COUNT(h1wk.TRANSACTION_TS)           AS IP_TXN_NUM_L1WK,
            COALESCE(SUM(h24.PURCHASE_AMOUNT),0) AS IP_TXN_AMT_L24H,
            COALESCE(SUM(h1wk.PURCHASE_AMOUNT),0)AS IP_TXN_AMT_L1WK,
            COUNT(DISTINCT h.CUSTOMER_ID)        AS IP_DISTINCT_CUST_L1H,
            COUNT(DISTINCT h24.CUSTOMER_ID)      AS IP_DISTINCT_CUST_L24H,
            COUNT(DISTINCT h1wk.CUSTOMER_ID)     AS IP_DISTINCT_CUST_L1WK
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS i
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h
            ON h.IP_ADDRESS = i.IP_ADDRESS AND h.TRANSACTION_TS >= DATEADD('hour', -1, i.TRANSACTION_TS) AND h.TRANSACTION_TS < i.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h6
            ON h6.IP_ADDRESS = i.IP_ADDRESS AND h6.TRANSACTION_TS >= DATEADD('hour', -6, i.TRANSACTION_TS) AND h6.TRANSACTION_TS < i.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
            ON h24.IP_ADDRESS = i.IP_ADDRESS AND h24.TRANSACTION_TS >= DATEADD('hour', -24, i.TRANSACTION_TS) AND h24.TRANSACTION_TS < i.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h48
            ON h48.IP_ADDRESS = i.IP_ADDRESS AND h48.TRANSACTION_TS >= DATEADD('hour', -48, i.TRANSACTION_TS) AND h48.TRANSACTION_TS < i.TRANSACTION_TS
        LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
            ON h1wk.IP_ADDRESS = i.IP_ADDRESS AND h1wk.TRANSACTION_TS >= DATEADD('day', -7, i.TRANSACTION_TS) AND h1wk.TRANSACTION_TS < i.TRANSACTION_TS
        GROUP BY i.txn_id
    )

    SELECT
        -- Transaction identifiers (excluded from feature_cols in cell 5)
        t.CUSTOMER_ID, t.MERCHANT_ID, t.WALLET_DPAN, t.IP_ADDRESS,
        t.TRANSACTION_TS, t.IS_FRAUD,

        -- Transaction attributes
        t.PURCHASE_AMOUNT,
        CASE WHEN t.MERCHANT_COUNTRY = 'GBR' THEN 1.0 ELSE 0.0 END AS IS_GBR,
        t.AUTHENTICATED_3DS_CHALLENGE_FLAG,
        t.IS_MERCHANT_INITIATED_PURCHASE,

        -- Time features
        HOUR(t.TRANSACTION_TS)                                         AS HOUR_OF_DAY,
        DAYOFWEEK(t.TRANSACTION_TS)                                    AS DAY_OF_WEEK,
        CASE WHEN DAYOFWEEK(t.TRANSACTION_TS) IN (0,6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        CASE WHEN HOUR(t.TRANSACTION_TS) < 5 THEN 1 ELSE 0 END        AS IS_NIGHT,

        -- Customer velocity (point-in-time correct self-joins)
        COUNT(h1.TRANSACTION_TS)                                       AS PURCHASES_NUM_L1H,
        COALESCE(SUM(h1.PURCHASE_AMOUNT), 0)                           AS PURCHASES_AMT_L1H,
        COALESCE(MAX(h1.PURCHASE_AMOUNT), 0)                           AS PURCHASES_MAX_L1H,
        COALESCE(MIN(NULLIF(h1.PURCHASE_AMOUNT, 0)), 0)                AS PURCHASES_MIN_L1H,
        COUNT(DISTINCT h1.MERCHANT_ID)                                 AS DISTINCT_MERCHANTS_L1H,
        COUNT(h6.TRANSACTION_TS)                                       AS PURCHASES_NUM_L6H,
        COALESCE(SUM(h6.PURCHASE_AMOUNT), 0)                           AS PURCHASES_AMT_L6H,
        COUNT(h24.TRANSACTION_TS)                                      AS PURCHASES_NUM_L24H,
        COALESCE(SUM(h24.PURCHASE_AMOUNT), 0)                          AS PURCHASES_AMT_L24H,
        COALESCE(MAX(h24.PURCHASE_AMOUNT), 0)                          AS PURCHASES_MAX_L24H,
        COALESCE(MIN(NULLIF(h24.PURCHASE_AMOUNT, 0)), 0)               AS PURCHASES_MIN_L24H,
        COUNT(DISTINCT h24.MERCHANT_ID)                                AS DISTINCT_MERCHANTS_L24H,
        COUNT(DISTINCT h24.WALLET_DPAN)                                AS DISTINCT_DPANS_L24H,
        COUNT(CASE WHEN h24.MERCHANT_COUNTRY = 'GBR' THEN 1 END)       AS PURCHASES_GBR_NUM_L24H,
        COUNT(h48.TRANSACTION_TS)                                      AS PURCHASES_NUM_L48H,
        COALESCE(SUM(h48.PURCHASE_AMOUNT), 0)                          AS PURCHASES_AMT_L48H,
        COUNT(h1wk.TRANSACTION_TS)                                     AS PURCHASES_NUM_L1WK,
        COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)                         AS PURCHASES_AMT_L1WK,
        COALESCE(MAX(h1wk.PURCHASE_AMOUNT), 0)                         AS PURCHASES_MAX_L1WK,
        COUNT(DISTINCT h1wk.MERCHANT_ID)                               AS DISTINCT_MERCHANTS_L1WK,
        COUNT(DISTINCT h1wk.WALLET_DPAN)                               AS DISTINCT_DPANS_L1WK,
        COUNT(DISTINCT h6.MERCHANT_ID)                                 AS DISTINCT_MERCHANTS_L6H,

        -- Derived ratio features (same formula as scoring service — ensures training/serving parity)
        DIV0(COUNT(h1.TRANSACTION_TS),  COUNT(h1wk.TRANSACTION_TS))    AS VELOCITY_RATIO_1H_L1WK,
        DIV0(COUNT(h6.TRANSACTION_TS),  COUNT(h1wk.TRANSACTION_TS))    AS VELOCITY_RATIO_6H_L1WK,
        DIV0(COUNT(h24.TRANSACTION_TS), COUNT(h1wk.TRANSACTION_TS))    AS VELOCITY_RATIO_24H_L1WK,
        DIV0(SUM(h1.PURCHASE_AMOUNT),   SUM(h1wk.PURCHASE_AMOUNT))     AS SPEND_BURST_1H_L1WK,
        DIV0(SUM(h6.PURCHASE_AMOUNT),   SUM(h1wk.PURCHASE_AMOUNT))     AS SPEND_BURST_6H_L1WK,
        DIV0(SUM(h24.PURCHASE_AMOUNT),  SUM(h1wk.PURCHASE_AMOUNT))     AS SPEND_BURST_24H_L1WK,
        -- Merchant concentration: low diversity = high concentration = suspicious
        1.0 - DIV0(COUNT(DISTINCT h24.MERCHANT_ID), NULLIF(COUNT(h24.TRANSACTION_TS),0)) AS MERCHANT_CONCENTRATION_24H,

        -- Customer profile (join to DIM_CUSTOMERS for static attributes)
        COALESCE(p.CUSTOMER_AGE, 30)              AS CUSTOMER_AGE,
        COALESCE(p.DAYS_SINCE_REGISTRATION, 365)  AS DAYS_SINCE_REGISTRATION,
        COALESCE(p.IS_HIGH_RISK, 0)               AS IS_HIGH_RISK,
        COALESCE(p.LIFETIME_TXN_COUNT, 0)         AS LIFETIME_TXN_COUNT,
        COALESCE(p.LIFETIME_TXN_TOTAL, 0)         AS LIFETIME_TXN_TOTAL,
        COALESCE(p.AVG_TXN_AMOUNT_30D, 0)         AS AVG_TXN_AMOUNT_30D,
        -- Profile-derived features
        CASE WHEN COALESCE(p.LIFETIME_TXN_COUNT, 0) = 0 THEN 1 ELSE 0 END    AS IS_FIRST_PURCHASE,
        CASE WHEN COALESCE(p.DAYS_SINCE_REGISTRATION, 999) <= 7  THEN 1 ELSE 0 END AS IS_NEW_ACCOUNT_7D,
        CASE WHEN COALESCE(p.DAYS_SINCE_REGISTRATION, 999) <= 30 THEN 1 ELSE 0 END AS IS_NEW_ACCOUNT_30D,
        -- Amount deviation from historical average
        DIV0(t.PURCHASE_AMOUNT - COALESCE(p.AVG_TXN_AMOUNT_30D, 0), NULLIF(p.AVG_TXN_AMOUNT_30D,0)) AS AMOUNT_PCT_DEVIATION,
        DIV0(t.PURCHASE_AMOUNT, NULLIF(COALESCE(MAX(h1wk.PURCHASE_AMOUNT), 0), 0)) AS AMOUNT_RATIO_TO_HIST_MAX,

        -- Merchant/DPAN/IP velocity (from CTEs)
        COALESCE(ma.MERCHANT_TXN_NUM_L1H,    0) AS MERCHANT_TXN_NUM_L1H,
        COALESCE(ma.MERCHANT_TXN_NUM_L6H,    0) AS MERCHANT_TXN_NUM_L6H,
        COALESCE(ma.MERCHANT_TXN_NUM_L24H,   0) AS MERCHANT_TXN_NUM_L24H,
        COALESCE(ma.MERCHANT_TXN_NUM_L48H,   0) AS MERCHANT_TXN_NUM_L48H,
        COALESCE(ma.MERCHANT_TXN_NUM_L1WK,   0) AS MERCHANT_TXN_NUM_L1WK,
        COALESCE(ma.MERCHANT_TXN_AMT_L1H,    0) AS MERCHANT_TXN_AMT_L1H,
        COALESCE(ma.MERCHANT_TXN_AMT_L24H,   0) AS MERCHANT_TXN_AMT_L24H,
        COALESCE(ma.MERCHANT_TXN_AMT_L1WK,   0) AS MERCHANT_TXN_AMT_L1WK,
        COALESCE(ma.MERCHANT_MAX_TXN_L24H,   0) AS MERCHANT_MAX_TXN_L24H,
        COALESCE(ma.MERCHANT_UNIQUE_CUST_L1H, 0) AS MERCHANT_UNIQUE_CUST_L1H,
        COALESCE(ma.MERCHANT_UNIQUE_CUST_L24H,0) AS MERCHANT_UNIQUE_CUST_L24H,
        COALESCE(ma.MERCHANT_UNIQUE_CUST_L1WK,0) AS MERCHANT_UNIQUE_CUST_L1WK,
        COALESCE(da.DPAN_TXN_NUM_L1H,    0) AS DPAN_TXN_NUM_L1H,
        COALESCE(da.DPAN_TXN_NUM_L6H,    0) AS DPAN_TXN_NUM_L6H,
        COALESCE(da.DPAN_TXN_NUM_L24H,   0) AS DPAN_TXN_NUM_L24H,
        COALESCE(da.DPAN_TXN_NUM_L48H,   0) AS DPAN_TXN_NUM_L48H,
        COALESCE(da.DPAN_TXN_NUM_L1WK,   0) AS DPAN_TXN_NUM_L1WK,
        COALESCE(da.DPAN_TXN_AMT_L24H,   0) AS DPAN_TXN_AMT_L24H,
        COALESCE(da.DPAN_TXN_AMT_L1WK,   0) AS DPAN_TXN_AMT_L1WK,
        COALESCE(da.DPAN_DISTINCT_CUST_L24H,0) AS DPAN_DISTINCT_CUST_L24H,
        COALESCE(da.DPAN_DISTINCT_CUST_L1WK,0) AS DPAN_DISTINCT_CUST_L1WK,
        COALESCE(ia.IP_TXN_NUM_L1H,    0) AS IP_TXN_NUM_L1H,
        COALESCE(ia.IP_TXN_NUM_L6H,    0) AS IP_TXN_NUM_L6H,
        COALESCE(ia.IP_TXN_NUM_L24H,   0) AS IP_TXN_NUM_L24H,
        COALESCE(ia.IP_TXN_NUM_L48H,   0) AS IP_TXN_NUM_L48H,
        COALESCE(ia.IP_TXN_NUM_L1WK,   0) AS IP_TXN_NUM_L1WK,
        COALESCE(ia.IP_TXN_AMT_L24H,   0) AS IP_TXN_AMT_L24H,
        COALESCE(ia.IP_TXN_AMT_L1WK,   0) AS IP_TXN_AMT_L1WK,
        COALESCE(ia.IP_DISTINCT_CUST_L1H,  0) AS IP_DISTINCT_CUST_L1H,
        COALESCE(ia.IP_DISTINCT_CUST_L24H, 0) AS IP_DISTINCT_CUST_L24H,
        COALESCE(ia.IP_DISTINCT_CUST_L1WK, 0) AS IP_DISTINCT_CUST_L1WK

    FROM (SELECT * FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (1000000 ROWS)) t

    -- Customer velocity: 5 self-joins for 1h, 6h, 24h, 48h, 7d windows
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1
        ON h1.CUSTOMER_ID = t.CUSTOMER_ID AND h1.TRANSACTION_TS >= DATEADD('hour', -1, t.TRANSACTION_TS) AND h1.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h6
        ON h6.CUSTOMER_ID = t.CUSTOMER_ID AND h6.TRANSACTION_TS >= DATEADD('hour', -6, t.TRANSACTION_TS) AND h6.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
        ON h24.CUSTOMER_ID = t.CUSTOMER_ID AND h24.TRANSACTION_TS >= DATEADD('hour', -24, t.TRANSACTION_TS) AND h24.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h48
        ON h48.CUSTOMER_ID = t.CUSTOMER_ID AND h48.TRANSACTION_TS >= DATEADD('hour', -48, t.TRANSACTION_TS) AND h48.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
        ON h1wk.CUSTOMER_ID = t.CUSTOMER_ID AND h1wk.TRANSACTION_TS >= DATEADD('day', -7, t.TRANSACTION_TS) AND h1wk.TRANSACTION_TS < t.TRANSACTION_TS

    -- Merchant / DPAN / IP velocity from CTEs
    LEFT JOIN merch_agg ma ON ma.txn_id = t.txn_id
    LEFT JOIN dpan_agg  da ON da.txn_id = t.txn_id
    LEFT JOIN ip_agg    ia ON ia.txn_id = t.txn_id

    -- Customer profile join
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE p ON p.CUSTOMER_ID = t.CUSTOMER_ID

    GROUP BY ALL
""")

training_df.write.save_as_table('FRAUD_DEMO_DEV.ML.TRAINING_SET', mode='overwrite')
elapsed = time.time() - start
row_count = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').count()
print(f'Training set materialised: {row_count:,} rows, {elapsed:.1f}s')
print('Features: customer/merchant/dpan/ip velocity (5 windows) + profile + derived')


## 3.2 Train XGBoost Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
import xgboost as xgb

training_pd = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').to_pandas()
print(f'Loaded {len(training_pd):,} rows, {training_pd.shape[1]} columns')

# Exclude identifiers, target, raw timestamp, and high-cardinality string columns.
# Everything else is a numeric feature — velocity counts, amounts, ratios, flags.
EXCLUDE = {
    'CUSTOMER_ID', 'MERCHANT_ID', 'WALLET_DPAN', 'IP_ADDRESS',
    'TRANSACTION_TS', 'IS_FRAUD',
}
feature_cols = [c for c in training_pd.columns if c not in EXCLUDE]
X = training_pd[feature_cols].fillna(0).astype(float)
y = training_pd['IS_FRAUD'].astype(int)

print(f'Features selected: {len(feature_cols)}')
print(f'  Customer velocity : {sum(1 for c in feature_cols if c.startswith("PURCHASES_") or c.startswith("DISTINCT_"))}')
print(f'  Merchant velocity : {sum(1 for c in feature_cols if c.startswith("MERCHANT_"))}')
print(f'  DPAN velocity     : {sum(1 for c in feature_cols if c.startswith("DPAN_"))}')
print(f'  IP velocity       : {sum(1 for c in feature_cols if c.startswith("IP_"))}')
print(f'  Profile/derived   : {sum(1 for c in feature_cols if c in ("CUSTOMER_AGE","DAYS_SINCE_REGISTRATION","IS_HIGH_RISK","LIFETIME_TXN_COUNT","LIFETIME_TXN_TOTAL","AVG_TXN_AMOUNT_30D"))}')

fraud_rate   = y.mean()
scale_pos    = int((1 - fraud_rate) / fraud_rate)
print(f'\nFraud rate: {fraud_rate:.4%} | scale_pos_weight: {scale_pos}')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    tree_method='hist',
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=42,
)

print('\nTraining XGBoost...')
t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
print(f'Training complete in {time.time()-t0:.1f}s')

y_prob  = model.predict_proba(X_val)[:, 1]
auc_pr  = average_precision_score(y_val, y_prob)
roc_auc = roc_auc_score(y_val, y_prob)
print(f'\nAUC-PR:  {auc_pr:.4f}  (primary metric for imbalanced fraud data)')
print(f'ROC-AUC: {roc_auc:.4f}')


## 3.3 Register in Model Registry

Registers the trained model and promotes it DEV → STAGING → PROD.

In [ ]:
# Initialise the Snowflake Model Registry — the versioned model store in FRAUD_DEMO_DEV.ML.
# The Registry is a Snowflake schema-level object that stores model artefacts, metrics,
# and metadata. It is not a separate service — it lives inside your Snowflake account.
reg = Registry(session=session, database_name='FRAUD_DEMO_DEV', schema_name='ML')

# sample_input_data is used by the Registry to infer the model's input schema.
# It must be a Pandas DataFrame with the same columns that will be sent at inference time.
sample_input = X_train.iloc[:100]

# Drop existing models if present — makes this cell idempotent for re-runs.
# PROD model may be in use by an inference service; skip if it can't be dropped.
for db in ['FRAUD_DEMO_DEV', 'FRAUD_DEMO_STAGING', 'FRAUD_DEMO_PROD']:
    try:
        session.sql(f'DROP MODEL IF EXISTS {db}.ML.FRAUD_DETECTION_MODEL').collect()
    except Exception:
        pass

# log_model() serialises the model, uploads it to the Registry, and records metrics.
# version_name='v1' — use semantic versioning for auditability.
# metrics — a dict of scalar evaluation metrics stored alongside the model artefact.
model_version = reg.log_model(
    model,
    model_name='FRAUD_DETECTION_MODEL',
    version_name='v1',
    sample_input_data=sample_input,
    metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc), 'scale_pos_weight': scale_pos},
    comment=f'XGBoost fraud classifier. {len(feature_cols)} features. Trained on transactions table (no DT dependency).',
)
print(f'Model registered: FRAUD_DEMO_DEV.ML.FRAUD_DETECTION_MODEL v1')
print(f'AUC-PR: {auc_pr:.4f}')

# Model promotion: DEV → STAGING → PROD.
# We log the same model object to each environment's Registry.
# If a version already exists (e.g., PROD is pinned by an inference service), skip gracefully.
for db, comment in [('FRAUD_DEMO_STAGING', 'Promoted from DEV after validation.'),
                    ('FRAUD_DEMO_PROD', 'Production model.')]:
    try:
        env_reg = Registry(session=session, database_name=db, schema_name='ML')
        env_reg.log_model(model, model_name='FRAUD_DETECTION_MODEL', version_name='v1',
            sample_input_data=sample_input,
            metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc)},
            comment=comment)
        print(f'  Promoted to {db}.ML.FRAUD_DETECTION_MODEL v1')
    except Exception as e:
        if 'already exist' in str(e).lower():
            print(f'  {db}: version v1 already exists — skipped (model may be in use by inference service)')
        else:
            raise

print('\nPromotion complete: DEV → STAGING → PROD')
print('\nNext: NB04 — deploy SPCS scoring endpoint')

In [ ]:
# Export model for the custom SPCS scoring service (nb04_serving.ipynb).
# The service loads these two files from the mounted MODEL_STAGE volume at startup.
#
# fraud_model.json  — XGBoost model in native JSON format (fast load, version-independent)
# feature_cols.json — ordered list of feature column names (ensures training/serving parity)
import json

model.save_model('/tmp/fraud_model.json')
with open('/tmp/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# PUT uploads to stage; auto_compress=False keeps the JSON readable
session.file.put('/tmp/fraud_model.json',  '@FRAUD_DEMO_PROD.ML.MODEL_STAGE', overwrite=True, auto_compress=False)
session.file.put('/tmp/feature_cols.json', '@FRAUD_DEMO_PROD.ML.MODEL_STAGE', overwrite=True, auto_compress=False)

print('Exported to @FRAUD_DEMO_PROD.ML.MODEL_STAGE:')
print('  fraud_model.json   — XGBoost model')
print(f'  feature_cols.json  — {len(feature_cols)} feature column names')
print('\nNext: nb04_serving.ipynb — build Docker image and deploy SPCS scoring service')
